# 07 · benchmarking — scorers, structural quality & loaders

Benchmark every interface scorer on real TCR-pMHC complexes, and check how close modelled
structures are to their ground truth. Given three directories of complex PDBs — **ground
truth**, **modelled** (same filenames), and **negatives** — the benchmark asks, per scorer:

1. **Agreement** — do modelled complexes score like their ground truth? (Pearson/Spearman)
2. **Discrimination** — do modelled *real* complexes score differently from *negatives*?
   (AUROC / AUPRC / Cohen's d on a leakage-aware test split)

Plus, independent of any scorer:

3. **Structural agreement** — Cα iRMSD over the 6 CDR loops → **HQ / MQ / AQ / LQ** tiers.

Full method: [`../docs/benchmarks.md`](../docs/benchmarks.md). Driver + comparison scripts:
[`../experiments/scorer_benchmarking/`](../experiments/scorer_benchmarking/) and
[`../experiments/loader_comparison/`](../experiments/loader_comparison/).

**Install**
```bash
pip install "kinapse[bench,structures]"
pip install "ifscore @ git+https://github.com/LilianDenzler/scoring_functions"
ifscore install all        # provision every scorer; `geometry_scoring` needs nothing
```

## 1. Structural agreement (model vs GT) — runnable here

`structural_agreement` IMGT-numbers both structures with the kinapse loader, identifies the
CDR/FR regions, and computes Cα RMSDs with the same Kabsch region-superposition used in the
ensemble analysis: per-CDR (framework-aligned), per-CDR local (loop-only), framework, and the
**Cα iRMSD over the 6 CDRs**. Here we compare the packaged example TCR to *itself* → identical
→ 0 Å → tier `HQ`.

In [ ]:
import os; os.environ.setdefault("ANARCI_CPU", "1")
from kinapse.benchmarks import structural_agreement, assign_tier

s = structural_agreement("data/example_tcr.pdb", "data/example_tcr.pdb")   # self vs self
print("Cα iRMSD over 6 CDRs:", s["struct__cdr_irmsd"], "Å  -> tier", assign_tier(s["struct__cdr_irmsd"]))
{k: v for k, v in s.items() if k.endswith("_rmsd") and "local" not in k}   # per-CDR + framework

`assign_tier` maps (Cα iRMSD, DockQ) → a quality tier. DockQ is optional — it's folded in
automatically when the `dockq` scorer is run, so HQ is enriched for high-confidence pairs.

| tier | Cα iRMSD | DockQ (if known) |
|---|---|---|
| **HQ** | ≤ 2 Å | ≥ 0.80 |
| **MQ** | ≤ 5 Å | ≥ 0.49 |
| **AQ** | < 5 Å | ≥ 0.23 |
| **LQ** | otherwise | |

In [ ]:
for irmsd, dockq in [(1.5, 0.9), (3.0, 0.6), (4.9, 0.3), (7.0, 0.9), (1.5, None)]:
    print(f"iRMSD={irmsd} Å, DockQ={dockq} -> {assign_tier(irmsd, dockq)}")

`tiers_summary` rolls the per-complex tiers into the HQ/MQ/AQ/LQ distribution over the
modelled positives (the validated pairs):

In [ ]:
import pandas as pd
from kinapse.benchmarks import tiers_summary
demo = pd.DataFrame([{"role": "model_pos", "tier": t} for t in ["HQ","HQ","MQ","AQ","LQ","HQ"]])
tiers_summary(demo)

## 2. The full scorer benchmark

Point it at three directories (modelled filenames must match the ground truth stems). It loads
each complex to derive correct chains (receptor = TCR α/β, ligand = pMHC), scores everything
with ifscore, makes a **leakage-aware** split (a complex's GT + model never straddle
train/test), and writes `scores.csv`, `agreement.csv`, `discrimination.csv`, `structural.csv`,
`tiers.csv`, `chains_cache.json`.

**Command line** (`-j` parallelises chain numbering *and* scoring; the run is resumable via
`chains_cache.json`):
```bash
python -m kinapse.benchmarks.scorer_benchmark \
    --gt    /path/TCR_complexes_openmm_minimised \
    --model /path/TCR_complexes_tfold_openmm_minimised \
    --neg   /path/negative_TCR_complexes_tfold/openmm_minimised \
    --out   scorer_benchmark_out --scorers all -j 16
# quick check (no provisioning): --limit 12 --scorers geometry_scoring
```

**Python:**
```python
from kinapse.benchmarks import run_scorer_benchmark
res = run_scorer_benchmark(gt_dir, model_dir, neg_dir, out_dir="out",
                           scorers="all", n_jobs=16)
res["agreement"]; res["discrimination"]; res["tiers"]
```

Or use the ready-made driver with this project's default paths:
[`../experiments/scorer_benchmarking/run_benchmark.py`](../experiments/scorer_benchmarking/run_benchmark.py)
(`--plots` also writes summary figures).

### GPU scorers (optional)

Two ifscore scorers use the GPU — `dproqa` (predicted DockQ) and `deeprank_gnn_esm`
(predicted fnat). They run in **CUDA containers**; ifscore adds `--gpus all` automatically,
so you set no device flag. Everything else (including the neural `esmif`/`proteinmpnn`) is
**CPU-only** — a GPU gives no speedup there, it only *adds* these two predictors.

They're included in `--scorers all` once their images are built:
```bash
ifscore install dproqa deeprank_gnn_esm    # from an ifscore source checkout (build context)
```
Requires Docker/Apptainer + the NVIDIA container toolkit. They are container-per-structure
and slow (~1.5–3 min/complex), so they lengthen a full run considerably — include them when
you want predicted DockQ/fnat, not for speed. See [`../docs/scoring.md`](../docs/scoring.md).

## 3. Loader backend — native or STCRpy

The loader that does **chain identification and CDR annotation** is pluggable (`--loader` /
`loader=`):

- **`native`** (default) — kinapse's own loader (ANARCII/ANARCI; no conda needed).
- **`stcrpy`** — OPIG **STCRpy** run as a *fully external* model (kinapse never imports it).
  Chains from `get_VA`/`get_VB` + antigen/MHC; CDRs from STCRpy's IMGT fragments. Both loaders
  emit the **same** region keys, so agreement / discrimination / tiers are computed identically.

STCRpy needs its own environment (ANARCI models, PLIP, OpenBabel). Point kinapse at it with
`KINAPSE_STCRPY_PYTHON` = the absolute path to *that env's* `python`:
```bash
micromamba create -n stcrpy python=3.10 -y && micromamba activate stcrpy
pip install stcrpy && ANARCI --build_models && pip install plip
export KINAPSE_STCRPY_PYTHON="$(which python)"     # the stcrpy env's python, not kinapse's

python -m kinapse.benchmarks.scorer_benchmark ... --loader stcrpy
```
Standalone (structural agreement for one pair, either backend):
```python
from kinapse.benchmarks import structural_agreement, assign_tier
s = structural_agreement(model_pdb, gt_pdb, loader="stcrpy")   # or loader="native"
assign_tier(s["struct__cdr_irmsd"])
```

## 4. Compare the two loaders

Two purpose-built experiments:

**a) Loading itself — do they identify the same chains and CDRs?**
[`../experiments/loader_comparison/`](../experiments/loader_comparison/) →
`compare_loading.py` runs both loaders on the same PDBs and reports chain set-equality and,
per CDR, the residue-set Jaccard + the Cα distance for residues both put in the same loop
(≈0 ⇒ same physical residue numbered the same).
```bash
export KINAPSE_STCRPY_PYTHON=/path/to/stcrpy-env/bin/python
python compare_loading.py --pdb-dir /path/TCR_complexes --limit 50 -j 8
```

**b) Benchmark outputs — does the choice of loader change the results?**
Run the benchmark once per loader into **separate** `--out` dirs (each has its own
`chains_cache.json`, so a shared dir would leak the other loader's chains), then diff:
```bash
python run_benchmark.py --scorers all -j 16 --out results_native
python run_benchmark.py --scorers all -j 16 --loader stcrpy --out results_stcrpy
python compare_loaders.py results_native results_stcrpy --labels native stcrpy
```
`compare_loaders.py` diffs coverage, agreement, discrimination, tiers, and per-complex
Cα-iRMSD (correlation, mean |Δ|, tier-agreement crosstab).